In [1]:
import os
import shutil
import cv2
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# Prevenim blocarea thread-urilor OpenCV
cv2.setNumThreads(0)

# ==========================================
# 1. CONFIGURARE CAI 
# ==========================================
# Sursa: Datasetul echilibrat de la pasul anterior
INPUT_BASE_DIR = Path(r"B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_augmented")

# Destinatia: Datasetul SUPREM, gata de bagat in PyTorch/TensorFlow
OUTPUT_BASE_DIR = Path(r"B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_augmented_bengraham")

CLASSES = ["0", "1", "2", "3", "4"]
SPLITS = ["train", "val", "test"]
WORKERS = max((os.cpu_count() or 2) - 1, 1)

# ==========================================
# 2. FUNCTIA BEN GRAHAM & HALO
# ==========================================
def procesare_ben_graham_elipsa_halo(image_path, size=512):
    img = cv2.imread(str(image_path))
    if img is None:
        return None

    img_resized = cv2.resize(img, (size, size))
    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)

    # 1. Extragerea conturului (scapa de colturile negre de la augmentari)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    kernel_clean = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    thresh_clean = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_clean)
    
    contours, _ = cv2.findContours(thresh_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return img_resized # Fallback daca nu gaseste contur

    contur_ochi = max(contours, key=cv2.contourArea)
    masca_ochi = np.zeros((size, size), dtype=np.uint8)
    cv2.drawContours(masca_ochi, [contur_ochi], -1, 255, thickness=cv2.FILLED)
    
    kernel_eroziune = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    masca_ochi = cv2.erode(masca_ochi, kernel_eroziune, iterations=1)

    # 2. Transformarea Ben Graham clasica
    sigmaX = size / 30.0 
    blur = cv2.GaussianBlur(gray, (0, 0), sigmaX)
    ben_graham_gray = cv2.addWeighted(gray, 4, blur, -4, 128)

    # 3. Halo exterior (Background 128 perfect)
    masca_exterior = cv2.bitwise_not(masca_ochi)
    dist = cv2.distanceTransform(masca_exterior, cv2.DIST_L2, 5)
    
    halo_width = 35.0 
    rezultat_final = np.full((size, size), 128.0, dtype=np.float32)
    
    halo_mask = (dist > 0) & (dist <= halo_width)
    fraction = dist[halo_mask] / halo_width
    rezultat_final[halo_mask] = 128.0 * fraction

    # Lipim ochiul in centru
    interior_mask = (masca_ochi == 255)
    rezultat_final[interior_mask] = ben_graham_gray[interior_mask]

    # Returnam in format BGR (3 canale) pentru arhitecturile de CNN
    rezultat_final = rezultat_final.astype(np.uint8)
    return cv2.cvtColor(rezultat_final, cv2.COLOR_GRAY2BGR)

def process_task(task):
    src_path, dst_path = task
    try:
        processed_img = procesare_ben_graham_elipsa_halo(src_path)
        if processed_img is not None:
            # Salvam in format PNG pentru a pastra calitatea intacta (fara artefacte JPEG pe gri)
            cv2.imwrite(str(dst_path), processed_img)
            return True, ""
        return False, f"Nu s-a putut citi: {src_path.name}"
    except Exception as e:
        return False, f"Eroare procesare {src_path.name}: {e}"

# ==========================================
# 3. EXECUTIA PRINCIPALA
# ==========================================
def main():
    if not INPUT_BASE_DIR.exists():
        print(f"Eroare: Folderul sursa {INPUT_BASE_DIR} nu exista!")
        return

    if OUTPUT_BASE_DIR.exists():
        print("Curatam folderul de output vechi...")
        shutil.rmtree(OUTPUT_BASE_DIR)

    tasks = []

    print("\n--- 1. Scanarea imaginilor de procesat ---")
    for split in SPLITS:
        for cls in CLASSES:
            input_dir = INPUT_BASE_DIR / split / cls
            output_dir = OUTPUT_BASE_DIR / split / cls
            
            if not input_dir.exists():
                continue
                
            output_dir.mkdir(parents=True, exist_ok=True)
            
            for img_path in input_dir.glob("*.*"):
                # Indiferent ce format e la intrare, la iesire il facem .png
                dst_path = output_dir / f"{img_path.stem}.png"
                tasks.append((img_path, dst_path))

    print(f"Total imagini pregatite pentru Ben Graham: {len(tasks)}")

    print(f"\n--- 2. Executia in paralel (Asta poate dura un pic) ---")
    processed, errors = 0, 0

    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        futures = [executor.submit(process_task, task) for task in tasks]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Procesare Ben Graham"):
            ok, msg = future.result()
            if ok: processed += 1
            else: 
                errors += 1
                print(f"\n[Eroare] {msg}")

    print("\n" + "="*50)
    print("FINALIZAT CU SUCCES! EXPERIMENTUL ESTE GATA.")
    print("="*50)
    print(f"Dataset-ul tau suprem se afla in:\n📂 {OUTPUT_BASE_DIR}")
    print(f"\nTotal imagini preprocesate reusit: {processed}")
    if errors > 0:
        print(f"Imagini care au dat eroare: {errors}")

if __name__ == '__main__':
    main()


--- 1. Scanarea imaginilor de procesat ---
Total imagini pregatite pentru Ben Graham: 72789

--- 2. Executia in paralel (Asta poate dura un pic) ---


Procesare Ben Graham:   0%|          | 0/72789 [00:00<?, ?it/s]


FINALIZAT CU SUCCES! EXPERIMENTUL ESTE GATA.
Dataset-ul tau suprem se afla in:
📂 B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_augmented_bengraham

Total imagini preprocesate reusit: 72789
